In [1]:
from pathlib import Path
import json

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

ROOT = Path.home() / "Desktop" / "FinCascade"

DATA_DIR = ROOT / "data"
GRAPH_DIR = ROOT / "graph"

with open(
    GRAPH_DIR / "graph.json",
    "r",
    encoding="utf-8"
) as f:
    graph_data = json.load(f)

metadata = pd.read_csv(
    DATA_DIR / "entities.csv"
)

asset_stats = pd.read_csv(
    DATA_DIR / "asset_statistics.csv"
)

ml_stress = pd.read_csv(
    DATA_DIR / "ml_stress_scores.csv",
    index_col=0,
    parse_dates=True
)

print("Graph nodes:", len(graph_data["nodes"]))
print("Graph edges:", len(graph_data["edges"]))
print("ML stress rows:", len(ml_stress))

Graph nodes: 26
Graph edges: 155
ML stress rows: 1377


In [2]:
DG = nx.DiGraph()

for node in graph_data["nodes"]:

    node_id = node["id"]

    attrs = {
        k: v
        for k, v in node.items()
        if k != "id"
    }

    DG.add_node(
        node_id,
        **attrs
    )


for edge in graph_data["edges"]:

    source = edge["source"]
    target = edge["target"]

    attrs = {
        k: v
        for k, v in edge.items()
        if k not in ["source", "target"]
    }

    DG.add_edge(
        source,
        target,
        **attrs
    )


print("Directed nodes:", DG.number_of_nodes())
print("Directed edges:", DG.number_of_edges())

Directed nodes: 26
Directed edges: 155


In [3]:
def get_edge_effect(attrs):

    lag_corr = attrs.get(
        "lag_correlation",
        None
    )

    corr = attrs.get(
        "correlation",
        None
    )

    # Prefer directional lag information
    if (
        lag_corr is not None
        and pd.notna(lag_corr)
        and lag_corr != 0
    ):
        effect = float(lag_corr)

    elif (
        corr is not None
        and pd.notna(corr)
    ):
        effect = float(corr)

    else:
        effect = float(
            attrs.get("weight", 0)
        )

    return np.clip(
        effect,
        -0.95,
        0.95
    )

In [12]:
# ==========================================
# STABLE FINANCIAL CONTAGION ENGINE
# ==========================================

# Pre-compute stable transmission coefficients
TRANSMISSION = {}

for source in DG.nodes():

    raw_edges = {}

    for target in DG.successors(source):

        raw_edges[target] = get_edge_effect(
            DG[source][target]
        )

    total_abs_strength = sum(
        abs(v)
        for v in raw_edges.values()
    )

    # Only normalize dense nodes.
    # Sparse economically meaningful links
    # (e.g. Crude -> ONGC) retain raw strength.
    scale = max(
        1.0,
        total_abs_strength
    )

    TRANSMISSION[source] = {
        target: effect / scale
        for target, effect
        in raw_edges.items()
    }


def simulate_shock(
    initial_shocks,
    max_hops=4,
    damping=0.55,
    min_propagation=0.0005,
    shock_cap=0.50
):

    total_impact = {
        node: 0.0
        for node in DG.nodes()
    }

    frontier = {}

    propagation_log = []

    # -------------------------------
    # Initial shock
    # -------------------------------
    for node, shock in initial_shocks.items():

        if node not in DG:
            raise ValueError(
                f"Unknown node: {node}"
            )

        shock = float(
            np.clip(
                shock,
                -shock_cap,
                shock_cap
            )
        )

        total_impact[node] = shock
        frontier[node] = shock

    # -------------------------------
    # Multi-hop diffusion
    # -------------------------------
    for hop in range(
        1,
        max_hops + 1
    ):

        next_frontier = {}

        for source, source_delta in frontier.items():

            for target, transmission_effect in (
                TRANSMISSION.get(
                    source,
                    {}
                ).items()
            ):

                propagated = (
                    source_delta
                    * transmission_effect
                    * damping
                )

                if abs(propagated) < min_propagation:
                    continue

                next_frontier[target] = (
                    next_frontier.get(
                        target,
                        0.0
                    )
                    + propagated
                )

                propagation_log.append({

                    "hop": hop,

                    "source": source,
                    "target": target,

                    "raw_edge_effect":
                        get_edge_effect(
                            DG[source][target]
                        ),

                    "transmission_effect":
                        transmission_effect,

                    "incoming_shock":
                        source_delta,

                    "propagated_shock":
                        propagated
                })

        # ---------------------------------
        # Bound each incremental shock
        # ---------------------------------
        for target in next_frontier:

            next_frontier[target] = float(
                np.clip(
                    next_frontier[target],
                    -shock_cap,
                    shock_cap
                )
            )

            total_impact[target] = float(
                np.clip(
                    total_impact[target]
                    + next_frontier[target],
                    -shock_cap,
                    shock_cap
                )
            )

        frontier = next_frontier

        if not frontier:
            break

    # -------------------------------
    # Results
    # -------------------------------
    results = pd.DataFrame({
        "ticker":
            list(total_impact.keys()),

        "shock":
            list(total_impact.values())
    })

    name_map = metadata.set_index(
        "ticker"
    )["name"].to_dict()

    sector_map = metadata.set_index(
        "ticker"
    )["sector"].to_dict()

    results["name"] = (
        results["ticker"]
        .map(name_map)
    )

    results["sector"] = (
        results["ticker"]
        .map(sector_map)
    )

    results["shock_pct"] = (
        results["shock"] * 100
    )

    results["absolute_impact_pct"] = (
        results["shock_pct"].abs()
    )

    results = (
        results
        .sort_values(
            "absolute_impact_pct",
            ascending=False
        )
        .reset_index(drop=True)
    )

    propagation_log = pd.DataFrame(
        propagation_log
    )

    return results, propagation_log

In [13]:
transmission_check = []

for source in DG.nodes():

    raw_sum = sum(
        abs(
            get_edge_effect(
                DG[source][target]
            )
        )
        for target in DG.successors(source)
    )

    normalized_sum = sum(
        abs(v)
        for v in TRANSMISSION.get(
            source,
            {}
        ).values()
    )

    transmission_check.append({
        "ticker": source,
        "name": DG.nodes[source].get(
            "name",
            source
        ),
        "raw_outgoing_strength": raw_sum,
        "normalized_strength": normalized_sum
    })


transmission_check = (
    pd.DataFrame(transmission_check)
    .sort_values(
        "raw_outgoing_strength",
        ascending=False
    )
)

display(
    transmission_check.head(10)
)

print(
    "\nMaximum normalized outgoing strength:",
    transmission_check[
        "normalized_strength"
    ].max()
)

,ticker,name,raw_outgoing_strength,normalized_strength
0,^NSEI,NIFTY 50,11.025267,1.0
1,^NSEBANK,Bank Nifty,7.611121,1.0
18,DLF.NS,DLF,6.285356,1.0
7,SBIN.NS,SBI,5.729687,1.0
25,BAJAJFINSV.NS,Bajaj Finserv,5.115590,1.0
24,BAJFINANCE.NS,Bajaj Finance,4.585583,1.0
8,AXISBANK.NS,Axis Bank,4.296977,1.0
6,ICICIBANK.NS,ICICI Bank,4.111985,1.0
5,HDFCBANK.NS,HDFC Bank,3.960812,1.0
19,GODREJPROP.NS,Godrej Properties,3.590252,1.0



Maximum normalized outgoing strength: 1.0000000000000002


In [14]:
oil_results, oil_paths = simulate_shock(
    {
        "CL=F": -0.20
    },
    max_hops=4
)

display(
    oil_results[
        [
            "ticker",
            "name",
            "sector",
            "shock_pct",
            "absolute_impact_pct"
        ]
    ].head(15)
)

,ticker,name,sector,shock_pct,absolute_impact_pct
0,CL=F,Crude Oil,Energy,-20.000000,20.000000
1,ONGC.NS,ONGC,Energy,-2.719719,2.719719
2,HINDUNILVR.NS,Hindustan Unilever,FMCG,1.428605,1.428605
3,INR=X,USD/INR,Macro,0.000000,0.000000
4,^NSEBANK,Bank Nifty,Banking,0.000000,0.000000
5,HDFCBANK.NS,HDFC Bank,Banking,0.000000,0.000000
6,ICICIBANK.NS,ICICI Bank,Banking,0.000000,0.000000
7,SBIN.NS,SBI,Banking,0.000000,0.000000
8,GC=F,Gold,Safe Haven,0.000000,0.000000
9,^NSEI,NIFTY 50,Market,0.000000,0.000000


In [6]:
name_map = metadata.set_index(
    "ticker"
)["name"].to_dict()

oil_paths["source_name"] = (
    oil_paths["source"].map(name_map)
)

oil_paths["target_name"] = (
    oil_paths["target"].map(name_map)
)

oil_paths["propagated_pct"] = (
    oil_paths["propagated_shock"]
    * 100
)

print("===== CRUDE OIL SHOCK SIMULATION =====")

print("\nInitial shock:")
print("Crude Oil: -20%")

print("\nEntities meaningfully affected:")
print(
    (
        oil_results[
            "absolute_impact_pct"
        ] >= 0.5
    ).sum()
)

print("\nONGC estimated total impact:")

display(
    oil_results[
        oil_results["ticker"]
        == "ONGC.NS"
    ][
        [
            "name",
            "shock_pct"
        ]
    ]
)

print("\nStrongest propagation events:")

display(
    oil_paths[
        [
            "hop",
            "source_name",
            "target_name",
            "edge_effect",
            "propagated_pct"
        ]
    ]
    .assign(
        abs_propagated=lambda x:
        x["propagated_pct"].abs()
    )
    .sort_values(
        "abs_propagated",
        ascending=False
    )
    .drop(
        columns="abs_propagated"
    )
    .head(15)
)

===== CRUDE OIL SHOCK SIMULATION =====

Initial shock:
Crude Oil: -20%

Entities meaningfully affected:
3

ONGC estimated total impact:


,name,shock_pct
1,ONGC,-2.719719



Strongest propagation events:


,hop,source_name,target_name,edge_effect,propagated_pct
0,1,Crude Oil,ONGC,0.247247,-2.719719
1,1,Crude Oil,Hindustan Unilever,-0.129873,1.428605


In [7]:
SCENARIOS = {

    "Oil Crash": {
        "CL=F": -0.20
    },

    "Banking Crisis": {
        "^NSEBANK": -0.15,
        "HDFCBANK.NS": -0.10,
        "ICICIBANK.NS": -0.10,
        "SBIN.NS": -0.10,
        "AXISBANK.NS": -0.10
    },

    "Tech Crash": {
        "TCS.NS": -0.15,
        "INFY.NS": -0.15,
        "HCLTECH.NS": -0.15,
        "WIPRO.NS": -0.15
    },

    "Market Crash": {
        "^NSEI": -0.12,
        "^NSEBANK": -0.15
    },

    "Currency Shock": {
        "INR=X": 0.08
    },

    "Rate Hike Proxy": {
        "^NSEBANK": -0.08,
        "BAJFINANCE.NS": -0.10,
        "BAJAJFINSV.NS": -0.10,
        "DLF.NS": -0.10,
        "GODREJPROP.NS": -0.10
    }
}

print("Available scenarios:")

for scenario in SCENARIOS:
    print("-", scenario)

Available scenarios:
- Oil Crash
- Banking Crisis
- Tech Crash
- Market Crash
- Currency Shock
- Rate Hike Proxy


In [16]:
latest_ml_risk = float(
    ml_stress["ml_market_risk_score"].iloc[-1]
)

max_pagerank = max(
    DG.nodes[n].get("pagerank", 0)
    for n in DG.nodes()
)


def calculate_systemic_risk(
    results,
    initial_shocks
):

    initial_nodes = set(
        initial_shocks.keys()
    )

    secondary = results[
        ~results["ticker"].isin(initial_nodes)
    ].copy()

    # --------------------------------
    # 1. Propagation intensity
    # --------------------------------
    mean_secondary_impact = (
        secondary["absolute_impact_pct"].mean()
    )

    propagation_score = min(
        (mean_secondary_impact / 5) * 100,
        100
    )

    # --------------------------------
    # 2. Contagion breadth
    # --------------------------------
    affected = (
        secondary["absolute_impact_pct"]
        >= 0.5
    ).sum()

    breadth_score = (
        affected
        / max(len(secondary), 1)
    ) * 100

    # --------------------------------
    # 3. Source systemic importance
    # --------------------------------
    source_scores = []

    for source in initial_nodes:

        pagerank = DG.nodes[source].get(
            "pagerank",
            0
        )

        normalized = (
            pagerank
            / max_pagerank
        ) * 100

        source_scores.append(
            normalized
        )

    centrality_score = (
        np.mean(source_scores)
        if source_scores
        else 0
    )

    # --------------------------------
    # 4. Current ML market context
    # --------------------------------
    ml_context_score = latest_ml_risk

    # --------------------------------
    # Final Systemic Risk Score
    # --------------------------------
    systemic_score = (
        0.35 * propagation_score
        +
        0.25 * breadth_score
        +
        0.20 * centrality_score
        +
        0.20 * ml_context_score
    )

    systemic_score = float(
        np.clip(
            systemic_score,
            0,
            100
        )
    )

    # --------------------------------
    # Updated risk interpretation
    # --------------------------------
    if systemic_score >= 65:
        level = "CRITICAL"

    elif systemic_score >= 45:
        level = "HIGH"

    elif systemic_score >= 25:
        level = "MODERATE"

    else:
        level = "LOW"

    return {
        "systemic_risk_score": systemic_score,
        "risk_level": level,

        "propagation_score": propagation_score,
        "breadth_score": breadth_score,
        "centrality_score": centrality_score,
        "ml_context_score": ml_context_score,

        "affected_entities": int(affected),

        "mean_secondary_impact_pct":
            mean_secondary_impact
    }

In [17]:
def run_scenario(
    scenario_name,
    shocks
):

    results, paths = simulate_shock(
        shocks,
        max_hops=4,
        damping=0.55
    )

    risk = calculate_systemic_risk(
        results,
        shocks
    )

    return {
        "scenario": scenario_name,
        "initial_shocks": shocks,
        "results": results,
        "paths": paths,
        "risk": risk
    }

In [18]:
scenario_outputs = {}
scenario_summary = []

for scenario_name, shocks in SCENARIOS.items():

    print("Running:", scenario_name)

    output = run_scenario(
        scenario_name,
        shocks
    )

    scenario_outputs[
        scenario_name
    ] = output

    risk = output["risk"]

    scenario_summary.append({
        "scenario":
            scenario_name,

        "systemic_risk_score":
            risk["systemic_risk_score"],

        "risk_level":
            risk["risk_level"],

        "affected_entities":
            risk["affected_entities"],

        "mean_secondary_impact_pct":
            risk["mean_secondary_impact_pct"],

        "propagation_score":
            risk["propagation_score"],

        "breadth_score":
            risk["breadth_score"]
    })


scenario_summary = pd.DataFrame(
    scenario_summary
).sort_values(
    "systemic_risk_score",
    ascending=False
).reset_index(drop=True)


print("\n===== SCENARIO RISK RANKING =====")

display(
    scenario_summary
)

Running: Oil Crash
Running: Banking Crisis
Running: Tech Crash
Running: Market Crash
Running: Currency Shock
Running: Rate Hike Proxy

===== SCENARIO RISK RANKING =====


,scenario,systemic_risk_score,risk_level,affected_entities,mean_secondary_impact_pct,propagation_score,breadth_score
0,Market Crash,50.325560,HIGH,12,0.699841,13.996829,50.000000
1,Banking Crisis,46.102673,HIGH,9,1.441028,28.820567,42.857143
2,Rate Hike Proxy,46.056751,HIGH,10,1.231924,24.638484,47.619048
3,Tech Crash,28.647373,MODERATE,1,0.736301,14.726018,4.545455
4,Oil Crash,20.533380,LOW,2,0.165933,3.318659,8.000000
5,Currency Shock,19.565467,LOW,0,0.000000,0.000000,0.000000


In [19]:
vulnerability_rows = []

for scenario_name, output in scenario_outputs.items():

    results = output[
        "results"
    ].copy()

    initial_nodes = set(
        output["initial_shocks"].keys()
    )

    for _, row in results.iterrows():

        vulnerability_rows.append({

            "scenario":
                scenario_name,

            "ticker":
                row["ticker"],

            "name":
                row["name"],

            "sector":
                row["sector"],

            "impact_pct":
                row["shock_pct"],

            "absolute_impact_pct":
                row["absolute_impact_pct"],

            "was_initial_shock":
                row["ticker"]
                in initial_nodes
        })


vulnerability_raw = pd.DataFrame(
    vulnerability_rows
)


# Focus on propagated vulnerability,
# excluding directly shocked sources
propagated_only = vulnerability_raw[
    vulnerability_raw[
        "was_initial_shock"
    ] == False
].copy()


vulnerability = (
    propagated_only
    .groupby(
        [
            "ticker",
            "name",
            "sector"
        ]
    )
    .agg(
        avg_impact_pct=(
            "absolute_impact_pct",
            "mean"
        ),

        max_impact_pct=(
            "absolute_impact_pct",
            "max"
        ),

        scenarios_affected=(
            "absolute_impact_pct",
            lambda x:
                (x >= 0.5).sum()
        )
    )
    .reset_index()
)


vulnerability["vulnerability_score"] = (
    0.60
    * vulnerability["max_impact_pct"]
    +
    0.40
    * vulnerability["avg_impact_pct"]
)


vulnerability = vulnerability.sort_values(
    [
        "scenarios_affected",
        "vulnerability_score"
    ],
    ascending=False
).reset_index(drop=True)


print(
    "===== MOST SYSTEMICALLY VULNERABLE ENTITIES ====="
)

display(
    vulnerability.head(15)
)

===== MOST SYSTEMICALLY VULNERABLE ENTITIES =====


,ticker,name,sector,avg_impact_pct,max_impact_pct,scenarios_affected,vulnerability_score
0,^NSEI,NIFTY 50,Market,5.084833,11.776357,3,9.099747
1,TMPV.NS,Tata Motors Passenger Vehicles,Auto,0.859553,2.139197,3,1.627339
2,M&M.NS,Mahindra & Mahindra,Auto,0.762058,1.817797,3,1.395501
3,RELIANCE.NS,Reliance,Energy,0.667843,1.516484,3,1.177027
4,MARUTI.NS,Maruti Suzuki,Auto,0.458215,0.926500,3,0.739186
5,DLF.NS,DLF,Realty,1.424843,5.069027,2,3.611353
6,BAJAJFINSV.NS,Bajaj Finserv,NBFC,1.250818,4.506061,2,3.203964
7,BAJFINANCE.NS,Bajaj Finance,NBFC,1.232228,4.492419,2,3.188342
8,ICICIBANK.NS,ICICI Bank,Banking,1.056159,3.355198,2,2.435582
9,SBIN.NS,SBI,Banking,0.919224,3.361267,2,2.384450


In [20]:
scenario_summary.to_csv(
    DATA_DIR / "scenario_summary.csv",
    index=False
)

vulnerability.to_csv(
    DATA_DIR / "vulnerability_ranking.csv",
    index=False
)

with open(
    DATA_DIR / "scenario_presets.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        SCENARIOS,
        f,
        indent=2
    )


for scenario_name, output in scenario_outputs.items():

    safe_name = (
        scenario_name
        .lower()
        .replace(" ", "_")
    )

    output["results"].to_csv(
        DATA_DIR / f"{safe_name}_results.csv",
        index=False
    )

    output["paths"].to_csv(
        DATA_DIR / f"{safe_name}_paths.csv",
        index=False
    )


print("===== NOTEBOOK 06 COMPLETE =====")

print("Scenario presets:", len(SCENARIOS))
print("Graph nodes:", DG.number_of_nodes())
print("Graph edges:", DG.number_of_edges())

print("\nSaved:")
print("- scenario_summary.csv")
print("- vulnerability_ranking.csv")
print("- scenario_presets.json")

print("\nSHOCK PROPAGATION ENGINE COMPLETE ✅")

===== NOTEBOOK 06 COMPLETE =====
Scenario presets: 6
Graph nodes: 26
Graph edges: 155

Saved:
- scenario_summary.csv
- vulnerability_ranking.csv
- scenario_presets.json

SHOCK PROPAGATION ENGINE COMPLETE ✅
